# Load All Wikipedia Clean Data Batches to DuckDB
This notebook loads all clean data batches from `/mnt/samsung/wiki-data-clean/` into a DuckDB database.

In [ ]:
import duckdb
import polars as pl
from pathlib import Path
from tqdm import tqdm

## Option 1: Direct Parquet Import (Fastest)

In [ ]:
# Connect to DuckDB (creates file if it doesn't exist)
# Database must be on a Linux filesystem (not exFAT) due to file locking requirements
db_path = '/home/steven/extraction-framework/wiki_data.duckdb'
con = duckdb.connect(db_path)

In [ ]:
# Create table from all parquet files in one command
# DuckDB will read all matching files
con.execute("""
    CREATE OR REPLACE TABLE wiki_articles AS 
    SELECT * FROM read_parquet('/mnt/samsung/wiki-data-clean/wiki_data_batch_*.parquet')
""")

# Check the result
result = con.execute("SELECT COUNT(*) as total_records FROM wiki_articles").fetchone()
print(f"Total records loaded: {result[0]:,}")

# Show table schema
print("\nTable Schema:")
print(con.execute("DESCRIBE wiki_articles").df())

In [4]:
# Check the result
result = con.execute("SELECT COUNT(*) as total_records FROM wiki_articles").fetchone()
print(f"Total records loaded: {result[0]:,}")

# Show table schema
print("\nTable Schema:")
print(con.execute("DESCRIBE wiki_articles").df())

Total records loaded: 4,030,002

Table Schema:
  column_name column_type null   key default extra
0       title     VARCHAR  YES  None    None  None
1        text     VARCHAR  YES  None    None  None


## Option 2: Batch-by-Batch Loading (More Control)

In [ ]:
# Alternative: Load one batch at a time with progress tracking
data_dir = Path('/mnt/samsung/wiki-data-clean')
parquet_files = sorted(data_dir.glob('wiki_data_batch_*.parquet'))

print(f"Found {len(parquet_files)} parquet files")

# Connect to DuckDB
con = duckdb.connect('/mnt/samsung/wiki_data.duckdb')

# Create table from first file
first_df = pl.read_parquet(str(parquet_files[0]))
con.execute("CREATE OR REPLACE TABLE wiki_articles AS SELECT * FROM first_df")
print(f"Created table with batch 1: {len(first_df):,} records")

# Insert remaining batches
for parquet_file in tqdm(parquet_files[1:], desc="Loading batches"):
    df_batch = pl.read_parquet(str(parquet_file))
    con.execute("INSERT INTO wiki_articles SELECT * FROM df_batch")

# Verify total count
total = con.execute("SELECT COUNT(*) FROM wiki_articles").fetchone()[0]
print(f"\nTotal records in database: {total:,}")


NameError: name 'Path' is not defined

## Create Indexes for Better Query Performance

In [ ]:
# Create indexes on commonly queried columns
# Adjust column names based on your actual schema

# Index on title for fast lookups
con.execute("CREATE INDEX IF NOT EXISTS idx_title ON wiki_articles(title)")

# Index on id if you have one
# con.execute("CREATE INDEX IF NOT EXISTS idx_id ON wiki_articles(id)")

print("Indexes created successfully")

Indexes created successfully


## Test Queries

In [ ]:
# Sample query: Get first 10 articles
result = con.execute("SELECT * FROM wiki_articles LIMIT 10").pl()
result

title,text
str,str
"""Anarchism""","""{{short description|Political …"
"""Albedo""","""{{Short description|Ratio of h…"
"""A""","""{{Short description|First lett…"
"""Alabama""","""{{Short description|U.S. state…"
"""Achilles""","""{{short description|Greek myth…"
"""Abraham Lincoln""","""{{Short description|President …"
"""Aristotle""","""{{Short description|Ancient Gr…"
"""An American in Paris""","""{{short description|Symphonic …"
"""Academy Award for Best Product…","""{{Short description|Academy Aw…"


In [ ]:
# Search by title (example)
search_term = 'Python'
result = con.execute(f"""
    SELECT title, LENGTH(text) as text_length 
    FROM wiki_articles 
    WHERE title LIKE '%{search_term}%' 
    LIMIT 20
""").pl()
result

title,text_length
str,i64
"""Monty Python's Life of Brian""",89757
"""Monty Python""",161361
"""Spam (Monty Python sketch)""",13076
"""The Spanish Inquisition (Monty…",8873
"""Monty Python and the Holy Grai…",47141
…,…
"""Template:Monty Python""",7839
"""Wikipedia:Articles for deletio…",1194
"""Colt Python""",20174


In [ ]:
# Get database statistics
stats = con.execute("""
    SELECT 
        COUNT(*) as total_articles,
        AVG(LENGTH(text)) as avg_text_length,
        MAX(LENGTH(text)) as max_text_length,
        MIN(LENGTH(text)) as min_text_length
    FROM wiki_articles
""").pl()
stats

In [ ]:
# Close connection when done
con.close()
print("Database connection closed")

## Database Info

- **Database Location**: `/mnt/samsung/wiki_data.duckdb`
- **Table Name**: `wiki_articles`
- **Access**: Use `duckdb.connect('/mnt/samsung/wiki_data.duckdb')` from any notebook/script
- **Query with Polars**: Use `.pl()` method to get Polars DataFrame from query results
- **Note**: Database file is on Linux filesystem (exFAT doesn't support file locking)